# ProAID — B1: CEFR Classifier Training (Stage A)
**PT-L2Detect Benchmark** | Google Colab + GPU T4

**Stage A improvements**: +AI training data (CEFR-aware labels) + Class weights

Fine-tune XLM-RoBERTa cho 5-class CEFR classification (A1-C1).
Dùng cả human + AI essays (AI labels là ground truth vì được sinh với CEFR-aware prompt).

## 0. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn torch accelerate matplotlib seaborn

In [ ]:
import json, os, sys, numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Upload Dataset
Upload `colab_data.zip` từ máy local.

In [ ]:
from google.colab import files
import zipfile

print("📤 Upload colab_data.zip")
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('data/')
        print(f'✅ Extracted {fname} → data/')

print(f'Files: {os.listdir("data/")}')

## 2. Load & Prepare Data (Stage A)
- Train = human + AI (1,781 essays)
- Val/Test = human only
- Class weights cho imbalance

In [ ]:
# === Constants ===
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1"]
CEFR2IDX = {l: i for i, l in enumerate(CEFR_LEVELS)}
IDX2CEFR = {i: l for i, l in enumerate(CEFR_LEVELS)}
NUM_LABELS = len(CEFR_LEVELS)

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load
train_data = load_jsonl('data/train_full.jsonl')
val_data   = load_jsonl('data/val_full.jsonl')
test_data  = load_jsonl('data/test_full.jsonl')

# === STAGE A: Train = ALL (human + AI), Val/Test = human only ===
train_all = train_data
val_human = [e for e in val_data if not e['is_ai']]
test_human = [e for e in test_data if not e['is_ai']]
train_human_only = [e for e in train_data if not e['is_ai']]

print(f'Train: {len(train_all)} (H:{sum(1 for e in train_all if not e["is_ai"])} + AI:{sum(1 for e in train_all if e["is_ai"])})')
print(f'Val:   {len(val_human)} (human-only)')
print(f'Test:  {len(test_human)} (human-only)')
print(f'Data increase: {len(train_human_only)} → {len(train_all)} (+{len(train_all)-len(train_human_only)})')

for name, ds in [('Train', train_all), ('Val', val_human), ('Test', test_human)]:
    c = Counter(e['cefr_level'] for e in ds)
    print(f'  {name}: {dict(sorted(c.items()))}')

# === Class weights ===
train_labels = [CEFR2IDX[e['cefr_level']] for e in train_all]
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_LABELS), y=train_labels)
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f'\n⚖️  Class weights: {dict(zip(CEFR_LEVELS, class_weights.round(3)))}')

## 3. Tokenization & DataLoaders

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CEFRDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        essay = self.data[idx]
        text = essay['text']
        label = CEFR2IDX[essay['cefr_level']]
        enc = self.tokenizer(text, truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_dataset = CEFRDataset(train_all, tokenizer, MAX_LENGTH)
val_dataset   = CEFRDataset(val_human, tokenizer, MAX_LENGTH)
test_dataset  = CEFRDataset(test_human, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

## 4. Model: XLM-RoBERTa + Classification Head

In [ ]:
class CEFRClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(out.last_hidden_state[:, 0, :])

model = CEFRClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training (Stage A)

In [ ]:
EPOCHS = 10
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
PATIENCE = 3

criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Steps: {total_steps} (warmup: {warmup_steps})')
print(f'Loss: CrossEntropyLoss + class weights')

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_total, preds, labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids, am, lb = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
            logits = model(ids, am)
            loss_total += criterion(logits, lb).item()
            preds.extend(torch.argmax(logits, -1).cpu().numpy())
            labels.extend(lb.cpu().numpy())
    return {
        'loss': loss_total / len(loader),
        'acc': accuracy_score(labels, preds),
        'wf1': f1_score(labels, preds, average='weighted'),
        'mf1': f1_score(labels, preds, average='macro'),
        'preds': preds, 'labels': labels
    }

def train_epoch(ep):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        ids, am, lb = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(ids, am), lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (step+1) % 20 == 0:
            print(f'  Ep {ep} | {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(train_loader)

# === RUN ===
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_wf1': []}
best_wf1, best_ep, patience_cnt = 0, 0, 0
os.makedirs('checkpoints', exist_ok=True)

print(f'{"="*60}')
print(f'STAGE A: {MODEL_NAME} — Train on Human+AI')
print(f'{"="*60}')

for epoch in range(1, EPOCHS+1):
    tl = train_epoch(epoch)
    vr = evaluate(model, val_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vr['loss'])
    history['val_acc'].append(vr['acc'])
    history['val_wf1'].append(vr['wf1'])
    print(f'  → Ep {epoch:2d} | T-Loss: {tl:.4f} | V-Loss: {vr["loss"]:.4f} | V-Acc: {vr["acc"]:.4f} | V-WF1: {vr["wf1"]:.4f}')
    if vr['wf1'] > best_wf1:
        best_wf1, best_ep, patience_cnt = vr['wf1'], epoch, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_wf1': vr['wf1'], 'model_name': MODEL_NAME, 'cefr_levels': CEFR_LEVELS},
                   'checkpoints/best_model.pt')
        print(f'  💾 Saved (WF1={best_wf1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'⏹ Early stop epoch {epoch}'); break

print(f'\n✅ Best Val WF1: {best_wf1:.4f} (epoch {best_ep})')

## 6. Human-only Baseline (để so sánh)

In [ ]:
print('Training human-only baseline for comparison...')
train_human_ds = CEFRDataset(train_human_only, tokenizer, MAX_LENGTH)
train_human_ldr = DataLoader(train_human_ds, batch_size=BATCH_SIZE, shuffle=True)

model_bl = CEFRClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)
opt_bl = AdamW(model_bl.parameters(), lr=LR)
steps_bl = len(train_human_ldr) * EPOCHS
sch_bl = get_linear_schedule_with_warmup(opt_bl, int(steps_bl*WARMUP), steps_bl)
criterion_bl = nn.CrossEntropyLoss()

best_bl, p_bl = 0, 0
for ep in range(1, EPOCHS+1):
    model_bl.train()
    for batch in train_human_ldr:
        ids, am, lb = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
        opt_bl.zero_grad()
        loss = criterion_bl(model_bl(ids, am), lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bl.parameters(), GRAD_CLIP)
        opt_bl.step(); sch_bl.step()
    vr = evaluate(model_bl, val_loader)
    if vr['wf1'] > best_bl: best_bl, p_bl = vr['wf1'], 0
    else: p_bl += 1
    if p_bl >= PATIENCE: break

test_bl = evaluate(model_bl, test_loader)
print(f'\n📊 COMPARISON:')
print(f'  Human-only (942 essays) → Test WF1: {test_bl["wf1"]:.4f}')
print(f'  Stage A   (1,781 essays) → Test WF1: {evaluate(model, test_loader)["wf1"]:.4f}')

del model_bl, train_human_ldr; torch.cuda.empty_cache()

## 7. Final Test Set Evaluation

In [ ]:
ckpt = torch.load('checkpoints/best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best model (Val WF1: {ckpt["val_wf1"]:.4f})')

test_results = evaluate(model, test_loader)
print(f'\n{"="*60}')
print(f'📊 TEST SET')
print(f'{"="*60}')
print(f'  Accuracy:    {test_results["acc"]:.4f}')
print(f'  Weighted F1: {test_results["wf1"]:.4f}')
print(f'  Macro F1:    {test_results["mf1"]:.4f}')
print(f'\n{classification_report(test_results["labels"], test_results["preds"], target_names=CEFR_LEVELS, digits=4)}')

status = '🎉 VERIFIED' if test_results['wf1'] >= 0.80 else f'⚠️ NOT MET (gap: {0.80-test_results["wf1"]:.4f})'
print(f'Claim C1 (≥0.80): {status}')

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_results['labels'], test_results['preds'])
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CEFR_LEVELS, yticklabels=CEFR_LEVELS, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            xticklabels=CEFR_LEVELS, yticklabels=CEFR_LEVELS, ax=axes[1])
axes[1].set_title('Normalized (Recall)', fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eps = range(1, len(history['train_loss'])+1)
axes[0].plot(eps, history['train_loss'], 'b-o', label='Train'); axes[0].plot(eps, history['val_loss'], 'r-s', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(eps, history['val_acc'], 'g-o', label='Val Acc'); axes[1].plot(eps, history['val_wf1'], 'm-s', label='Val WF1')
axes[1].axhline(y=0.80, color='r', linestyle='--', label='0.80 target')
axes[1].set_title('Validation'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

## 10. Error Analysis & Experiment Log

In [ ]:
print("🔍 Top confusions:\n")
errors = [(CEFR_LEVELS[i], CEFR_LEVELS[j], cm_norm[i,j], cm[i,j])
          for i in range(5) for j in range(5) if i!=j and cm_norm[i,j]>0.05]
errors.sort(key=lambda x: -x[2])
for tl, pl, rate, cnt in errors:
    print(f'  {tl} → {pl}: {rate*100:.1f}% ({int(cnt)})')

per_class_recall = cm.diagonal() / cm.sum(axis=1)
print(f'\n📊 Per-class recall:')
for i, l in enumerate(CEFR_LEVELS):
    m = ' ⚠️' if per_class_recall[i]==min(per_class_recall) else ''
    print(f'  {l}: {per_class_recall[i]:.4f} ({int(cm[i,i])}/{int(cm[i].sum())}){m}')

# Save log
experiment_log = {
    'experiment': 'B1.1 — CEFR Classifier Stage A',
    'timestamp': datetime.now().isoformat(),
    'model': MODEL_NAME, 'device': str(DEVICE),
    'train_size': len(train_all), 'train_human': len(train_human_only),
    'train_ai': sum(1 for e in train_all if e['is_ai']),
    'val_size': len(val_human), 'test_size': len(test_human),
    'hyperparameters': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
                        'max_length': MAX_LENGTH, 'warmup': WARMUP, 'wd': WEIGHT_DECAY,
                        'class_weights': {l: float(w) for l,w in zip(CEFR_LEVELS, class_weights)}},
    'results': {'test_acc': float(test_results['acc']), 'test_wf1': float(test_results['wf1']),
                'test_mf1': float(test_results['mf1']),
                'per_class_recall': {l: float(per_class_recall[i]) for i,l in enumerate(CEFR_LEVELS)}},
    'baselines': {'word_count_only_wf1': 0.2854, 'xlmr_human_only_wf1': 0.40},
    'stage': 'A — +AI data + Class weights'
}
with open('experiment_log.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_log, f, indent=2, ensure_ascii=False)
print(f'\n💾 experiment_log.json saved')

## 11. Download Results

In [ ]:
!zip -j b1_cefr_stageA.zip checkpoints/best_model.pt experiment_log.json confusion_matrix.png training_curves.png
files.download('b1_cefr_stageA.zip')